# 06. Entradas do Modelo 3 (Bayesian GPLVM)

Etapa 6 do fluxo: `01_download.ipynb` → `02_serie_modelos.ipynb` → `03_guia_bases_modelos.ipynb` → `04_entradas_modelo1_gp.ipynb` / `05_entradas_modelo2_rgp.ipynb` / `06_entradas_modelo3_gplvm.ipynb` → notebooks dos modelos (`07`–`09`).

Gera e salva a entrada do GPLVM a partir da `serie` salva no `02_serie_modelos.ipynb`. **Não ajusta o modelo.** O GPLVM recebe janelas de dias de cada estação, montadas dentro de cada bloco (nunca atravessam dois blocos); células sem valor visível viram 0 e a máscara as exclui da verossimilhança. Detalhes na seção 4.3 do `03_guia_bases_modelos.ipynb`.

Saída: `data/exports/<NOME>/modelo3_gplvm/janelas.parquet` (uma linha por janela) e `config.json`.

## 1. Carregar os módulos

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for folder in (start.resolve(), *start.resolve().parents):
        if (folder / "src" / "main.py").is_file() and (folder / "data" / "usgs").is_dir():
            return folder
    raise FileNotFoundError("Não encontrei src/main.py e data/usgs. Abra o notebook na pasta do projeto.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import EXPORT_ROOT
from src.model_data import split_summary
from src.model_exports import gplvm_table, gplvm_arrays, save_inputs
from src.model_inputs import select_split

## 2. Ler a base

In [ ]:
NOME = "serie_2010_2025_200est"   # o mesmo NOME do 02_serie_modelos.ipynb

serie = pd.read_parquet(EXPORT_ROOT / f"{NOME}.parquet")
print("Base:", (EXPORT_ROOT / f"{NOME}_config.json").read_text(encoding="utf-8"))
print(f"{len(serie):,} linhas | {serie['sensor_id'].nunique()} estações")

## 3. Montar e conferir a entrada

Uma linha por janela, com `sensor_id`, `split` e, para cada posição `k` da janela:

| Colunas | Conteúdo | Uso no GPLVM |
|---|---|---|
| `t_0 … t_{k}` | dia de cada célula | mapear o escore de volta para os dias |
| `y_0 … y_{k}` | valor (0 onde não é visível) | dados observados `Y` |
| `mask_0 … mask_{k}` | 1 nas células visíveis | quais células entram na verossimilhança |
| `true_0 … true_{k}` | valor verdadeiro (NaN sem valor) | **só para a métrica** |
| `hidden_0 … hidden_{k}` | 1 nas células escondidas | onde se mede a reconstrução |

Janelas com mais de `MAX_FALTANTE` células sem valor visível são descartadas, assim como a sobra no fim de cada bloco (menos dias que a janela).

In [ ]:
JANELA = 7          # dias por janela (7 = semanal; 30 = mensal)
MAX_FALTANTE = 0.5  # descarta janelas com mais de 50% de células sem valor visível

entradas = gplvm_table(serie, window=JANELA, max_missing=MAX_FALTANTE)

gl = gplvm_arrays(entradas)
print("janelas:", len(entradas), "| Y", gl["Y"].shape, " NaN:", np.isnan(gl["Y"]).any())
display(entradas.groupby("split", sort=False).agg(janelas=("sensor_id", "size"),
                                                  estacoes=("sensor_id", "nunique")))

## 4. Salvar

Se a entrada já existir com as mesmas escolhas, nada é regravado. Se as escolhas mudarem (outra janela, por exemplo), a célula para com erro: apague a pasta `data/exports/<NOME>/modelo3_gplvm/` para gerar de novo.

In [ ]:
save_inputs(entradas, NOME, "modelo3_gplvm", config={
    "estacoes": sorted(entradas["sensor_id"].unique().tolist()),
    "janela": JANELA,
    "max_faltante": MAX_FALTANTE,
})

## 5. Como o notebook do modelo lê esta entrada

```python
from src.model_exports import load_inputs, gplvm_arrays
from src.model_inputs import select_split, windows_to_days

entradas, cfg = load_inputs("serie_2010_2025_200est", "modelo3_gplvm")
gl = gplvm_arrays(entradas)

tr, te = select_split(gl, "treino"), select_split(gl, "teste")
Y_train, M_train = tr["Y"], tr["mask"]             # ajuste
Y_test, M_test = te["Y"], te["mask"]               # inferir q(X) e calcular o escore
Y_test_true, H_test = te["Y_true"], te["hidden"]   # só para a métrica

# escore por janela -> por dia: windows_to_days(te, elbo_negativo_por_janela)
```

`gplvm_arrays` devolve o mesmo dicionário que `gplvm_inputs`.